In [3]:
from ptlpinns.odes import equations, numerical
from ptlpinns.models import model, transfer
import numpy as np
import time
import torch

Note: computational time scales with N for PINNs but not for the numerical solvers

In [4]:
N_ITER = 800

N = 512
t_span = (0, 10)
t_eval = np.linspace(t_span[0], t_span[1], N)

RK45_time, Radau_time, PTL_PINN_inverting, PTL_PINN_not_inverting = [], [], [], []

In [5]:
zeta_list = [0, 0.4, 0.6, 10, 30]

w_list_transfer = [1, 1, 1, 1, 1]

forcing_names = ['']

def forcing(numpy=False):
    if not numpy:
        def force(t):
            return torch.stack((torch.zeros_like(t), torch.zeros_like(t)), dim=1)
    else:
        def force(t):
            return np.stack((np.zeros_like(t), np.zeros_like(t)), axis=1)
    return force

forcing_list = [forcing(True), forcing(True), forcing(True), forcing(True), forcing(True), forcing(True)]

def zeroes_1D(t):
    return np.zeros_like(t)

forcing_1D = [zeroes_1D, zeroes_1D, zeroes_1D, zeroes_1D, zeroes_1D, zeroes_1D]

ic_list = [[1, 0], [1, 0], [1, 0], [1, 0], [1, 0]]
p_list = [5, 5, 5, 5, 5]
epsilon_list = [0.5, 0.5, 0.5, 0.5, 0.5]

### RK45 and Radau times

In [6]:
total_RK45, total_Radau = [], []

for i, zeta in enumerate(zeta_list):

    ode = equations.ode_oscillator_1D(w_0=w_list_transfer[i], zeta=zeta, forcing_1D=forcing_1D[i], q=[(3, 1)], epsilon=0.5)

    print("solving for zeta =", zeta)
    RK45_time_list, Radau_time_list = [], []

    for j in range(N_ITER):

        start_RK45 = time.perf_counter()
        RK45_sol = numerical.solve_ode_equation(ode, t_span, t_eval, ic_list[i], method="RK45", rtol=1e-5, atol=1e-5)[0]
        end_RK45 = time.perf_counter()

        start_Radau = time.perf_counter()
        numerical.solve_ode_equation(ode, t_span, t_eval, ic_list[i], method="Radau", rtol=1e-5, atol=1e-5)[0]
        end_Radau = time.perf_counter()

        RK45_time_list.append(end_RK45 - start_RK45)
        Radau_time_list.append(end_Radau - start_Radau)

    total_RK45.append(RK45_time_list)
    total_Radau.append(Radau_time_list)

    Radau_time.append(np.mean(RK45_time_list[i]))
    RK45_time.append(np.mean(Radau_time_list[i]))

solving for zeta = 0
solving for zeta = 0.4
solving for zeta = 0.6
solving for zeta = 10
solving for zeta = 30


In [7]:
print("RK45 time:", RK45_time)
print("Radau time:", Radau_time)

RK45 time: [np.float64(0.5799693581648171), np.float64(0.30151091096922755), np.float64(0.37642843276262283), np.float64(0.02146997582167387), np.float64(0.041816330049186945)]
Radau time: [np.float64(0.11946127377450466), np.float64(0.005398923065513372), np.float64(0.01910401601344347), np.float64(0.07773291924968362), np.float64(0.5951052349992096)]


### PTL-PINNs

In [8]:
undamped_path = "/home/jovyan/PTL-PINNs/ptlpinns/models/train/undamped_k12"
undamped_name = "model_undamped_k12.pth"
undamped_model, _ = model.load_model(undamped_path, undamped_name)

underdamped_path = "/home/jovyan/PTL-PINNs/ptlpinns/models/train/underdamped_k12"
underdamped_name = "model_underdamped_k12.pth"
underdamped_model, _ = model.load_model(underdamped_path, underdamped_name)

overdamped_path = "/home/jovyan/PTL-PINNs/ptlpinns/models/train/overdamped_k12"
overdamped_name = "model_overdamped_k12.pth"
overdamped_model, _ = model.load_model(underdamped_path, underdamped_name)

12 True True True 1.0 16 [256, 256, 512]
12 True True True 1.0 16 [128, 128, 256]
12 True True True 1.0 16 [128, 128, 256]


In [ ]:
# Compute latent representation: H(t) and derivatives
H_dict_undamped = transfer.compute_H_dict(undamped_model, N=N, bias=True, t_span=(t_span[0], t_span[1]))
H_dict_underdamped = transfer.compute_H_dict(underdamped_model, N=N, bias=True, t_span=(t_span[0], t_span[1]))
H_dict_overdamped = transfer.compute_H_dict(overdamped_model, N=N, bias=True, t_span=(t_span[0], t_span[1]))

training_log = {'w_ode': 1.5, 'w_ic': 1}

In [ ]:
total_inverting, total_not_inverting = [], []

for i in range(len(zeta_list)):

    print("solving for zeta =", zeta_list[i])

    inverting, not_inverting = [], []

    for j in range(N_ITER):

        if zeta_list[i] == 0:
            solver = "LPM"
            H_dict = H_dict_undamped
        elif 0 < zeta_list[i] < 1:
            solver = "standard"
            H_dict = H_dict_underdamped
        else:
            solver = "standard"
            H_dict = H_dict_overdamped 

        # invert = True
        _, _, TL_time_inverting = transfer.compute_perturbation_solution([w_list_transfer[i]], [zeta_list[i]], [epsilon_list[i]], [p_list[i]],
                                                                [ic_list[i]], [forcing_list[i]], H_dict,
                                                                t_eval, training_log, all_p=True, comp_time=True,
                                                                solver=solver, w_sol = [], invert=True)
        
        # invert = False
        _, _, TL_time_not_inverting = transfer.compute_perturbation_solution([w_list_transfer[i]], [zeta_list[i]], [epsilon_list[i]], [p_list[i]],
                                                                [ic_list[i]], [forcing_list[i]], H_dict,
                                                                t_eval, training_log, all_p=True, comp_time=True,
                                                                solver=solver, w_sol = [], invert=False)
        
        inverting.append(TL_time_inverting[0])
        not_inverting.append(TL_time_not_inverting[0])

    total_inverting.append(inverting)
    total_not_inverting.append(not_inverting)

    PTL_PINN_inverting.append(np.mean(total_inverting[i]))
    PTL_PINN_not_inverting.append(np.mean(total_not_inverting[i]))

In [ ]:
for i in range(len(zeta_list)):
    print(f"zeta: {zeta_list[i]} | RK45: {RK45_time[i]} | Radau: {Radau_time[i]} | PTL-PINN: {PTL_PINN_inverting[i]} | PTL-PINN no invert: {PTL_PINN_not_inverting[i]}")

### H initialization — inference timing

Confirm that inference time is independent of how H is initialised (pretrained vs
random vs orthogonal).  Only the `invert=False` (no M-inversion) case is measured
because the cost of matrix-vector multiplication dominates and is identical for all
three H variants.

In [ ]:
# ── Load training logs (needed for compute_H_dict_random_init architecture) ───
_, tlog_undamped    = model.load_model(undamped_path,    undamped_name)
_, tlog_underdamped = model.load_model(underdamped_path, underdamped_name)

# ── Precompute random and orthogonal H for each model ─────────────────────────
H_dict_undamped_random    = transfer.compute_H_dict_random_init(tlog_undamped,    N=N, bias=True, t_span=t_span)
H_dict_underdamped_random = transfer.compute_H_dict_random_init(tlog_underdamped, N=N, bias=True, t_span=t_span)
H_dict_overdamped_random  = transfer.compute_H_dict_random_init(tlog_underdamped, N=N, bias=True, t_span=t_span)

H_dict_undamped_orth    = transfer.compute_H_dict_orthogonal(H_dict_undamped_random)
H_dict_underdamped_orth = transfer.compute_H_dict_orthogonal(H_dict_underdamped_random)
H_dict_overdamped_orth  = transfer.compute_H_dict_orthogonal(H_dict_overdamped_random)

# ── Timing loop ───────────────────────────────────────────────────────────────
time_random_list, time_orth_list = [], []

for i in range(len(zeta_list)):
    print("solving for zeta =", zeta_list[i])

    if zeta_list[i] == 0:
        solver = "LPM"
        hd_random = H_dict_undamped_random
        hd_orth   = H_dict_undamped_orth
    elif 0 < zeta_list[i] < 1:
        solver = "standard"
        hd_random = H_dict_underdamped_random
        hd_orth   = H_dict_underdamped_orth
    else:
        solver = "standard"
        hd_random = H_dict_overdamped_random
        hd_orth   = H_dict_overdamped_orth

    t_random, t_orth = [], []
    for j in range(N_ITER):
        _, _, tl_r = transfer.compute_perturbation_solution(
            [w_list_transfer[i]], [zeta_list[i]], [epsilon_list[i]], [p_list[i]],
            [ic_list[i]], [forcing_list[i]], hd_random,
            t_eval, training_log, all_p=True, comp_time=True,
            solver=solver, w_sol=[], invert=False)
        _, _, tl_o = transfer.compute_perturbation_solution(
            [w_list_transfer[i]], [zeta_list[i]], [epsilon_list[i]], [p_list[i]],
            [ic_list[i]], [forcing_list[i]], hd_orth,
            t_eval, training_log, all_p=True, comp_time=True,
            solver=solver, w_sol=[], invert=False)
        t_random.append(tl_r[0])
        t_orth.append(tl_o[0])

    time_random_list.append(np.mean(t_random))
    time_orth_list.append(np.mean(t_orth))

print()
print(f"{'zeta':<8} {'Pretrained':>14} {'Random H':>14} {'Orthogonal H':>14}")
print("-" * 52)
for i in range(len(zeta_list)):
    print(f"{zeta_list[i]:<8} {PTL_PINN_not_inverting[i]:>14.6f}"
          f" {time_random_list[i]:>14.6f} {time_orth_list[i]:>14.6f}")